In [ ]:
%%capture
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

without understanding tokens and embeddings we cannot understand how LLMs works see fig 2-1

LLM TOKENIZATION

the model generates its response in one token at a time, but tokens arent only the outputs but also the input which the model sees, a text form is broken into tokens before sending it to the model

HOW tokenizers prepare the inputs for LLMs

if we see  a high level view LLMS take an input prompt and then generate a response
but thats not what happens inside, the input prompt has to go through a tokenizer which breaks the input into pieces , see fig 2-3

to further understand this lets download an LLM and see how can we tokenize the input prompt



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

we will first write our prompt then tokenize it and give the tokens to the model, now we will tell the model to generate only 20 new tokens

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
input_ids=input_ids,
max_new_tokens=20

)
# Print the output
print(tokenizer.decode(generation_output[0]))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


everything after subject: is the modles 20 tokens output, the model didnt actually got the raw prompt but the tokenizers took the raw prompt and returned the information needed by the model in the form of input_ids variable, which the model used as its input

In [ ]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


this shows the LLMs respond to a series of numbers as shown in fig 2-4, each one is unique id of the specific token, the token can be a word, a char, or part of the word, these ids reference a table inside the tokenizer which contains all the tokens it knows

if we want to see the actual words which these ids hold we can use the decode method by tokenizer

In [ ]:
for id in input_ids[0]:
  print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


some tokens are complete words while some are part of words, punctuation char are their own tokens

there are no tokens for spaces and partial tokens like "izing" and "ic" have a special hidden char at their beginning which says they are connected with token that precedes them in text, tokens without that special char are assumed to have a space before them

if we move towards the output side we can also see the tokens generated by model for outputs using generation_output variable, it will show us the input tokens as well as the output tokens

In [ ]:
print(generation_output)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')


after 32001 all the tokens are of output, we can see the actual text for output side too using tokenizers decode method

In [ ]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


HOW DOES THE TOKENIZER BREAKDOWNS THE TEXT ?


it follows three major factors for this :

1. when we design the model we can choose the type of tokenization we want there are several methods like BYTE PAIR ENCODING (BPE) used in GPT models and WordPiece used in BERT, these methods are good at what they are designed to do which is to optimize an efficient set of tokens to represent a text dataset, but they do that in differnet ways

2. after deciding the method we neeed to choose a number of tokenizer design choices like vocab size and which special tokens to use

3. the tokenizer needs to be trained on the specific dataset to make the best vocab which it can use to represent the dataset, and even if we use the same params and methods a tokenizer trained on the english dataset will perform differently then the one trained on the code dataset or multiliinguial text dataset

tokenizers are also used during output gen, when the model returns an output token id, tokenizer is used to return the word associated with it as seen in fig 2-5

BYTE, CHAR, SUNWORDS and WORDS TOKENS


The tokenization way showed above is subword tokenization it is the most commonly used tokenization scheme, there 4 in total as shown in fig 2-6

1. word tokens
it was a common approach with methods like word2vec but now its becoming less popular in NLP, but they are still used in recommendation systems
there is one problem with word tokenizer, it cannot deal with new words that enter the dataset after the tokenizer is trained, and the vocabulary formed also have a lot of tokens for the words which have very minimum differences like apologize, apology, apologetic
this challange was solved by subword tokenization, it have a token for the apalog and then the suffix tokens like -y, -ize, -ogy, -etic, that are common with many other tokens as well this results in an efficient and more expressive vocab

2. subword tokens
this method contains partial words and full words as discussed arlier, it also have a benefit of represeting new words by breaking the new token into smaller chars, which will eventually be already present in the vocab

3. character tokens
this method can also deal with new words as it has raw letters to fall back on, this makes it robust and easier to tokenize but the modelling becomes more difficult for eg : if we use subword tokenization it will save "cat" as single token, but a model using char level tokens need to model the information to spell "c-a-t" while modelling the rest of the input too
and sub words tokens also have one more advantage, it can fit more text within the limited context length of a transformer model, subword tokens often average 3 chars per token

4. byte tokens
this method breaks down tokens into individual bytes that are used to represent the unicode chars, this method can be a competitive one in multilingual scenarios

some subword tokenizer methods also includes bytes as tokens in their vocab, they use it when they encounter chars they cant represent otherwise, the GPT 2 and Roberta tokenizers do this but this doesnt make them a tokenization free byte level tokenizers as they only represent a subset using this

Comparing Trained LLM Tokenizers

we will compare some tokenizers now, the newer tokenizers have changed their behvior to improve models performance, further we will see how code generation models or specialized models need specialized tokenizers

In [ ]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

the above example will help us to understand how exactly diff tokenizers work in different conditions like :

1. capitalization
2. languages other than english
3. emojis
4. programming, keywords, whitespaces(for identation in python)
5. num and digits
6. special tokens. these tokens have other roles than representing a text, like indicating the beginning of the text or the end of text(this way model can signal the system that it has completed the generation) or some other functions as well

we will start from the old tokenizers, each token will have different colour using this function

In [ ]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

BERT BASED MODEL (UNCASED)

tokenization method : word piece
vocab size : 30.522

special tokens:

1. unk_token [UNK]:
it is an unknown token given to the char or words for which tokenizer doesnt have any specific encoding

2. sep_token [SEP]
a seperator that enables certain tasks which requires giving the modle two tasks (in such cases the model is called cross encoder), eg reranking

3. pad_token [PAD]
it is a padding token which the model uses to fill several unused positions in the input as the model expects certain length of input, its context size

4. cls_token [CLS]
special token for classification tasks

5. mask_token [MASK]
it is used to mask or hide certain tokens during the training process

In [ ]:
show_tokens(text, "bert-base-uncased")

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

BERT was released with two major methods, cased where capitalization was kept and uncased where capitalization was converted to small cap letters, uncase is more popular, this what we see in uncase:

1. there is no newline breaks which means the model will be blind to the information encoded in the form of new lines for eg in chat log where each turn is a new line

2. all the text is in lowercase

3. the word capitalization is broken into two subtokens: capital and ##ization, the ## is used to indicate this token has another token which precedes it, this also tells us where the spaces are as it is assumed that tokens which dont have ## in front of them have spaces before them

4. the emoji and the chinese chars are replaced by [UNK] TOKEN which is a special token

BERT BASED MODEL (CASED):

tokenization method: wordpiece

voacab size: 28,996

special tokens: same as before

In [ ]:
show_tokens(text, "bert-base-cased")

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

it differs mainly in how it handles capitalization by including it

1. CAPITALIZATION is represented as 8 tokens now CA ##PI ##TA ##L ##I ##Z ##AT ##ION

2. both the methods in BERT wraps the input inside starting [CLS] and ending [SEP] token, they are utility tokens used to wrap the input text, [CLS] is a classification token used during sentence classification and [SEP] is a seperator token used for seperating two sentences in applications that requires passing two sentences to the model

GPT-2 (2019)

tokenization method : byte pair encoding (BPE)

vocab size: 50,257

special tokens: <|endoftext|>

1. we now have newline breaks rep in tokenizer

2. capitalization is preserved and we have 4 tokens for CAPITALIZATION

3. the special characters now have a unique token, they all looks same but they get encode with different token id so when we decode it using the token id we will be getting diff chars only

4. the two tabs are represented with two tokens, the four spaces are represented using the 3 tokens, where the final space is part of the the closing quote char


white space chars are important for model in understanding or generating the code, a model which uses a single token to represent the four spaces is more fine tuned to a python code dataset, but it can still work with four diff tokens but it will make the modelling task more difficult as the model keeps a record of identation level, which leads to worse performance
this is where tokenization choices help the model to improve on a certain task

In [ ]:
show_tokens(text, "gpt2")


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

FLAN-T5

tokenization method: SentencePiece, it is a simple lang independent subword tokenizer and detokenizer for neural text processing, it supports BPE and unigram lang model

vocab size: 32,100

special tokens: unk_token < UNK >, pad_token < PAD >

1. no newline or whitespace tokens, coding will be challenging here

2. again the chinese and the emoji are replaced by < unk > making the model completely blind to them

In [ ]:
show_tokens(text, "google/flan-t5-small")

English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600 </s> 

GPT-4

tokenization method: BPE

vocab size : little over 100,00
special tokens:
1. <|endoftext|>

2. fill in the middle tokens, they enable the LLM to generate a completion given not only the text before it but also considering the text after it
<|fim_prefix|>
<|fim_middle|>
<|fim_suffix|>

there are few differences between it and its previous version GPT 2

1. the gpt 4 uses a single token to represent the 4 whitespaces, infact it has a specific token for every sequence of whitespaces, till 83 whitespaces

2. they python keyword elif now has its own token, making it more suitable for coding tasks in addition to NLP

3. it uses fever tokens for most of the words, like CAPITALIZATION got wind up in just 2 tokens vs 4 tokens

In [ ]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")


 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 12 . 0 * 50 = 600 
 

STAR CODER 2

star coder 2 is a 15 billion parameter model focused on genrating code

tokenization method: BPE

vocab size: 49,152

special tokens: same as gpt 4
when we write code managing the context is very important as there may be many events where one function located in one file call another function located in some another file so the models need some way to being able to identify code located in different files in same repo while making a distinction for code located in another repo,
this is why star coder use some more special tokens for file name and repo name
1. < filename >
2. < reponame >
3. < gh_stars >

a major difference here is that for each digit there is a seperate token unlike seen in all other models till now, there is a hypothesis which says it leads to better rep of numbers and mathematics

In [ ]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")


 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

GALACTICA

GALACTICA model is made for scientific purposes it is trained on vast knowledge bases, many scientific papers, ref materials, it focuses more on tokenization, this makes it more sensitive to the specifics of the dataset its representing like it has special tokens for reasoning, citations, mathematics, amino acid seq and DNA seq

tokenization method: BPE

vocab size: 50,000

special tokens:
1. < s >
2. < pad >
3. < /s >
4. < unk >
5. refrences and citations are wrapped within two special type of tokens: [START_REF], [END_REF]
6. step by step reasoning, < work > is used for chain of thought reasoning

it is almost similar to starcoder 2, but this is the only tokenizer so far which assigns a single token to the string made up of two tabs


In [ ]:
show_tokens(text, "facebook/galactica-1.3b")


 English  and  CAP ITAL IZATION 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : "      "  Three  t abs :   "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

PHI-3 (and LLAMA 2)

it reuses the tokenizer of LLAMA 2 but adds a no of special tokens also

tokenization method: BPE

vocab size = 32,000

special tokens:
1. <|endoftext|>
2. chat tokens, with the rise of coversation type LLMs since 2023, tokenizers were modified by the addition of tokens that can indicate turns in conv and the roles of each speaker, these special tokens include: <|user|>, <|assistant|>, <|system|>

In [ ]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

TOKENIZER PROPERTIES


till now we have seen various methods for tokenization, but how can we define their behaviour ?, there are 3 major design choices for this, it will decide how tokenizer will break down the text, they are : the tokenization method, the initialization params and the domain of the data the tokenizer is specifically targetting

1. tokenizer methods:
the BPE is the most popular method, each of these methods have their own algorithms to decide how to choose the appropriate set of tokens to represent the dataset

2. tokenizer params:
now we have to choose some params for tokenizer like <BR>
VOCABULARY SIZE: how many tokens to include in tokenizer vocab<BR>
SPECIAL TOKENS: these are on us, we can deicde how many special tokens the model should track, we can add as many of them depending on the type of LLM we want to build, common choices are beginning of text token, end of text token, padding token, unknown token, CLS token, masking token <br>
CAPITALIZATION: we can decide how can we handle the capitalization, as capitalization contains some useful info often, but should we waste vocab space to include all caps version of words ?, we have to decide this

3. the domain of the data:
even if we select the same methods and params the tokenizer will behave different only, it will be based on the dataset it was trained on<br>
for example in a coding model the text focused tokenizer will tokenize the indentation spaces like this (see eg) but that can e=be improved by making different tokenization choices (see eg), the tokenization choices makes the models job easy, and yields to higher probab of improving

TOKEN EMBEDDINGS


now we know how one part of understanding the language works for the LLMs, language is a sequence of tokens and if we train a model on a large enough set of tokens it will start capturing some complex patterns that appeared during the training

1. if we train using a lot of english texts the model will be able to understand and generate english language

2. and if the training data contains factual information like from wikipedia it would be more capable of generating factual information, but the llms alone are not good enough for language models, this led to the discovery of RETRIEVAL AUGMENTED GENERATION (RAG), this approach combines search and LLMs

the next part of understanding the language is finding the best numerical representations to represent these embeddings so that the model could use them to calculate and model the text patterns, this means it helps the model in getting a clarity about a language or caoability to code or any other capabilities we expect from the model <br>
and this is what embeddings are, a numeric representation space that is used to capture the meanings and patterns in language

A LANGUAGE MODEL HOLDS EMBEDDINGS FOR THE VOCAB OF ITS TOKENIZER

when the tokenizers training finishes, it is then used for the models training, this is the reason pretrained LMs comes with their own tokenizer and cannot use any other one without training

the language models holds an embedding vector for each token present in the tokenizers vocab, as seen in fig 2-7, when we download a model a portion of the model is this embeddings matrix holding all these vectors

they are assigned random values initially like the model weights before training, when the training process starts it gets assign the values that will be useful for the specific purpose the model is trained to perform



CREATING CONTEXTUALIZED WORD EMBEDDINGS WITH LANGUAGE MODELS

now we know the embeddings are the input for the model, lets see how the model can create better token embedings, this is one of the ways to use language models for text representation like named-entity recognition or extractive text summarization, meaning summarizing the long text with some important points from the passage itself, instead of generating new words for summary

now instead of representing each token or word with a static embedding, language models create contextualized embeddings see fig 2-8, this will represent a word with a different token based on its context, these vectoes can be used for a number of purposes other than text applications, like AI image generation systems like DALL-E, midjourney, stable diffusion

eg:<br>
the model we will be using is deberta v3, it is a good embedding model, small and highly efficient, the code will load the tokenizer and model then use them to process the string "hello world"


In [ ]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

model.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

the output is saved in output variable and the dimensions of the variable is printed, if we ignore the first dim, we can read it as 4 tokens each one is embedded in a vector of 384 values, the first dim is just batch dimension used in training cases when we want to send multiple input sentences to the model at the same time, they are processed at the same time which will speed up the process

In [ ]:
output.shape

torch.Size([1, 4, 384])

lets see what these four vectors, is it that the two words got broken into 4 tokens or is it something else, we will see that using the decode method

In [ ]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


the tokenizer uses the CLS and SEP token at the start and end of the string and below is the output of the processed input text:

this is the raw outputs in top of this only applications for LLMs are build, the switch from token id to raw embeddings is the first step that takes place inside the LLM fig 2-9 shows the whole process

In [ ]:
output

tensor([[[-3.4805,  0.0862, -0.1818,  ..., -0.0610, -0.3909,  0.3022],
         [ 0.1885,  0.3201, -0.2313,  ...,  0.3721,  0.2471,  0.8057],
         [ 0.2089,  0.5010, -0.0495,  ...,  1.2197, -0.2277,  0.8574],
         [-3.4277,  0.0635, -0.1426,  ...,  0.0658, -0.4358,  0.3826]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

TEXT EMBEDDINGS (FOR SENTENCES AND WHOLE DOCUMENTS)

token embeddings are the main component for the LLM to perform, but a no of LLM applications requires operating of entire sentences, text documents, this is the reason special language models are made which can produce text embeddings which is a single vector to represent a piece of text longer than just one token

it takes an entire text and produces a single vector for its representation, that captures the meaning in some useful form, fig 2-10 shows that process

there are many ways to produce text embeddings, one of the most popular way is to average all the token embeddings produced, but high quality text en=mbedding model are there and trained specifically for text embeddings

text embeddings can be produced using sentence-transformers a library which leverages the power of embedding models, we will be using all-mpnet-base-v2 model for now as embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

the number of values or dimensions of the embedding vector depends on the underlying embedding model

in our case we got a single vector with dimension of 768 numerical values, these text embeddings can be used for categorization to semantic search to RAG

In [ ]:
vector.shape

(768,)

In [ ]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 20.0 MB/s eta 0:00:00


WORD EMBEDDINGS BEYOND LLMS


embeddings can be useful outside of language or text generation tasks, embeddings can also be meaningful vector representations of objects, therefore useful in recommnender engines and robotics

USING PRETRAINED WORD EMBEDDINGS

we will download the pretrained embeddings like word2vec or GLoVe using the gensim library:

In [ ]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


we now have embeddinggs for a large no of words trained on wikipedia, we will explore the embedding space by seeing the neighbours for specific words lets say "king"

In [ ]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

THE WORD2VEC ALGORITHM AND CONTRASIVE TRAINING

word2vec is trained using examples generated from text just like LLMs, lets say we have the text <br>  "Thou shalt not make a machine in the
likeness of a human mind” <br>
the algorithm uses a sliding window to generate the train examples, lets say we have a sliding window of size 2, this means we will consider 2 neighbors around the center word

the embeddings are gen using classification task, this task uses neural networks to predict whether the words commonly appears in the same context or not , think it as neural netorks which take two words and outputs 1 if they appear in the same context and 0 if they not

see fig 2-11, in the first sliding window we have 4 training examples

in each training window, the word present in the center is given as first input and then every other neighbour words are given as second input see fig 2-12, the final trained model should be able to identify the neighbour relationship and output 1 if two inputs are neighbours

we will also add negative examples so that the model dont just cheat by saying 1 for each one of them see fig 2-13

we dont need to put much pressure on thinking about the negative examples, infact many good models are able to identify only the positive samples from the randomly generated negative sample, you just need to identify the positive ones, so we can get random words and add them to the dataset and indicate they are not neighbours the model will output 0 when it sees them

we have 2 main concepts for word2vec now, skipgram method of selecting neighboring words, and random sampling which is adding randomly negative examples from the dataset

with this we can generate millions of training instances. Before proceeding to train on the dataset using neural networks we need to make some tokenization choices just like we did for LLM tokenizer like handling capitalization, punctuation, vocab size

now for each token an embedding will be randomly initialize. see 2-15, this will be a matrix of dim vocab_size x embeddding _dimensions

the model then take two embedding vectors and decides whether they are neigbours or not see fig 2-16

based on the prediction of the model the embeddings are updated so that the models correctness gets increase, and with the end of training we have a goated set of embeddings for all tokens in vocab



EMBEDDINGS FOR RECOMMENDATON SYSTEMS

<br>
embeddings are also used for recommendation task <br>
<br>
RECOMENDDING SONGS BY EMBEDDINGS<br>

we will use word2vec now to show how can we embed songs using human made playlists, each song will be treated as a word and each playlist will be a sentence see fig 2-17,  these embeddings can then be used for recommending songs which are appearing together in playlists



TRAINING A SONG EMBEDDING MODEL

first load the dataset

In [ ]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

lets see now what playlists list contain:

each elements inside it is a playlist containing a list of song ids

In [ ]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

train the model:

In [ ]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

now we will use the embeddings to find similar songs, the below code will list out all the songs which are similar to 2172

In [ ]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('2849', 0.9976382851600647),
 ('5586', 0.996614933013916),
 ('6624', 0.9962846040725708),
 ('3116', 0.9962383508682251),
 ('3167', 0.9958386421203613),
 ('3094', 0.9950186610221863),
 ('1922', 0.9945260882377625),
 ('3079', 0.9945164322853088),
 ('6658', 0.9941666126251221),
 ('2014', 0.994138777256012)]

2172 song is: this is a metallica song so the recommendations will be heavy metal type only

In [ ]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


the below code will decode the recommendations

In [ ]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
2849,Run To The Hills,Iron Maiden
5586,The Last In Line,Dio
6624,Everybody Wants Some!!!,Van Halen
3116,Communication Breakdown,Led Zeppelin
3167,Unchained,Van Halen


In [ ]:
print_recommendations(842)

,title,artist
id,,
886,Heartless,Kanye West
6741,Love In This Club (w\/ Young Jeezy),Usher
5698,Turnin' Me On (w\/ Lil Wayne),Keri Hilson
330,Hate It Or Love It (w\/ 50 Cent),The Game
27078,Out Of My Head (w\/ Trey Songz),Lupe Fiasco
